# L2a: Introduction to Functions and Interfaces

Functions turn a mathematical rule into a callable interface. We will progress from a direct Fibonacci function to typed models and multiple dispatch, while making input and mutation behavior explicit.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Define a function as an interface:__ Identify the core pieces every function definition shares across languages: a name, parameters, a body, and return behavior. Write a documented Julia function with typed arguments, a declared return type, and early returns that settle the base cases, so a caller can use it correctly without reading its body.
> * __Validate inputs and handle errors:__ Reject an input outside the documented domain with an assertion inside the function, and decide at the call site whether an error should stop the program or be handled. A try-catch block serves both purposes explored in this lecture: capturing the error message as a string for inspection, and recovering so the program continues past the failure with a fallback value.
> * __Select implementations with multiple dispatch:__ Separate a public method that validates input and manages state from internal methods that perform the calculation, and let the types of the arguments choose among them rather than branching on type inside one function. The exclamation-point naming convention marks the public method as mutating, and because keyword arguments do not participate in dispatch, the fieldless marker type selects the implementation only when passed as a positional argument.

Let's get started!
___

## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
# Load this meeting's file-relative environment, source code, and imports.
include(joinpath(@__DIR__, "Include.jl"));

Besides Julia's `Base` library, `Include.jl` loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which the rest of the week uses. This lecture checks its own results with [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) instead.

___

## Anatomy of a Function
A mathematical function $f:A\rightarrow B$ maps inputs in a domain $A$ to outputs in a codomain $B$. In a program, a function is a named block of code that can receive inputs and perform a task. It may return a value, produce a side effect such as printing or modifying data, or do both.

Every function definition has the same core pieces: a **name**, **parameters**, a **body**, and **return behavior**. The syntax varies by language:

| Language | Definition | Body | Return and types |
|:--|:--|:--|:--|
| **Julia** | `function f(x)` ... `end`, or `f(x) = expression` | Delimited by `end` | `return` is optional; the last expression is returned. Type annotations are optional. |
| **MATLAB/Octave** | `function y = f(x)` ... `end` | Delimited by `end` | Return values are named in the definition and assigned in the body. Types are dynamic. |
| **Python** | `def f(x):` | Indented | Values are returned with `return`. Type hints are optional and are not enforced at runtime. |
| **C** | `ReturnType f(ArgumentType x)` | Enclosed in `{}` | Argument and return types are declared; a non-`void` function returns a compatible value. |

The companion [Julia](examples/function_anatomy.jl), [Python](examples/function_anatomy.py), [C](examples/function_anatomy.c), and [Octave](examples/fahrenheit_to_celsius.m) files implement the same temperature-conversion function. See the [examples README](examples/README.md) for command-line and REPL instructions.

Now let's examine these pieces in Julia.
___

## Example: Starter Fibonacci Sequence Function
Let's look at a simple implementation of a function that computes the Fibonacci sequence given an integer $n\in\mathbb{Z}_{\geq{0}}$ as input. A [Fibonacci sequence](https://en.wikipedia.org/wiki/Fibonacci_sequence) is composed of the Fibonacci numbers $F_{n}$:
$$
\begin{align*}
F_{0} & = 0 \quad n = 0\\
F_{1} & = 1 \quad n = 1\\
F_{n} & = F_{n-2} + F_{n-1}\quad{n\geq{2}}
\end{align*}
$$

Here's an example function that computes the Fibonacci numbers in Julia. It starts [with the function documentation (don't forget the documentation!)](https://docs.julialang.org/en/v1/manual/documentation/#Writing-Documentation), then defines the function name and parameters, and finally implements the logic to compute the Fibonacci number and returns the result.

In [ ]:
"""
    fibonacci(n::Int64) -> Dict{Int64, Int64}

Return `F_0` through `F_n` in a dictionary keyed by Fibonacci index.

### Arguments
- `n::Int64`: Largest Fibonacci index to compute. It must be nonnegative.

### Returns
- `Dict{Int64, Int64}`: A mapping from every index `k` in `0:n` to `F_k`.
"""
function fibonacci(n::Int64)::Dict{Int64, Int64}
    
    # Enforce the nonnegative domain required by the Fibonacci recurrence.
    @assert n >= 0 "Hmmm. Major Malfunction - the argument `n` must be a non-negative integer";

    # Use mathematical Fibonacci indices as keys, so F_k is stored at sequence[k].
    sequence = Dict{Int64, Int64}();

    # Complete the two base cases before applying a recurrence that needs two
    # preceding values.
    if n == 0 
        sequence[0] = 0;
        return sequence;
    elseif n == 1
        sequence[0] = 0;
        sequence[1] = 1;
        return sequence;
    end

    # For n >= 2, seed the recurrence with F_0 and F_1.
    sequence[0] = 0;
    sequence[1] = 1;

    # Materialize the range 2:n as a vector, then compute F_i from the two
    # dictionary entries at the preceding mathematical indices.
    test_range = range(2, stop=n, step=1) |> collect;
    for i ∈ test_range
        sequence[i] = sequence[i-1] + sequence[i-2]
    end

    # Return the complete index-to-value mapping.
    return sequence;
end;

Call the function with an integer argument $n\in\mathbb{Z}_{\geq{0}}$, and store the result in a `fibonacci_dictionary::Dict{Int,Int}` variable. This dictionary maps the integers to their corresponding Fibonacci numbers.

In [ ]:
# Compute F_0 through F_10 for the reference-value checks below.
fibonacci_dictionary = fibonacci(10)

__Check__: Starting at 0, we know that the Fibonacci numbers are 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, ... . Let's check these values against our results stored in the `fibonacci_dictionary::Dict{Int64,Int64}` dictionary using [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert).

> __Interesting__: We have multiple conditions to check, i.e., the computed values for $F_{0},F_{1},\dots,F_{10}$ are correct. We've implemented this by enclosing [the `@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) in a `for` loop. This is a common pattern, but not the only (or most efficient) way we could have done this.

Do we pass the tests?

In [ ]:
let 
    
    # Store F_0 through F_10 in a one-based vector of reference values.
    true_results = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55];

    # Dictionary key i is the mathematical index, while vector position i + 1
    # holds the same reference value because Julia arrays are one-based.
    for i ∈ 0:10
        @assert fibonacci_dictionary[i] == true_results[i+1] "Fibonacci number at index $i is incorrect"
    end

    # Print the success message only after every index-value invariant holds.
    println("All tests passed!")
end

What happens if we pass a negative integer to the function? The [`@assert` macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) in our implementation should reject it and throw [an `AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError).

> __Why catch it here?__ An uncaught error stops the notebook at this cell. A stack trace, the report of the sequence of calls leading up to an error, is worth learning to read, but we want the rest of the notebook to run. So we wrap the call in [a try-catch block](https://docs.julialang.org/en/v1/base/base/#try) and capture the message as a string instead.

What does the error say?

In [ ]:
# The try expression returns the normal-path marker or the captured error text.
expected_error = try
    fibonacci(-12)
    "no error"
catch error
    sprint(showerror, error)
end

Capturing the message is one use of `try-catch`. The other is __recovery__: carrying on with a fallback when the call fails. Below we try to assign the result to `fibonacci_dictionary_2`, and the program keeps running past the failure rather than stopping:

In [ ]:
# Preserve this empty fallback unless the validated call returns successfully.
fibonacci_dictionary_2 = Dict{Int64,Int64}();
try
    # The negative index throws before the assignment can replace the fallback.
    fibonacci_dictionary_2 = fibonacci(-10)
catch e
    println("Caught an error: $e")
end
println("Program continues after handling the error.")

So what is in `fibonacci_dictionary_2` now? The call threw before it could return a value, so the assignment never happened and the variable still holds the empty dictionary it was initialized with. The type is intact; the contents are not:

In [ ]:
# Pass the fallback to typeof to confirm that its concrete dictionary type remains intact.
fibonacci_dictionary_2 |> typeof

___

## Example: Complex Fibonacci Sequence Function
Functions can be more complex, with multiple parameters, different types of return values, and even side effects. Let's extend our Fibonacci function to take composite types, optional parameters, and reimagine how we structure the code by introducing a key concept called encapsulation.

To start, let's define several composite types to represent the Fibonacci sequence and its properties, and how we wish to implement the function.

In [ ]:
# Separate sequence state from the marker types used to select loop implementations.
abstract type AbstractSequenceModel end
abstract type AbstractIterationModel end

"""
    MyFibonacciSequenceModel <: AbstractSequenceModel

Mutable state for a Fibonacci calculation.

### Fields
- `n::Int64`: The largest index to compute, so the sequence runs from `F₀` to `Fₙ` and holds `n + 1` entries.
- `sequence::Dict{Int64, Int64}`: The sequence itself, stored as a dictionary with indices as keys and Fibonacci numbers as values.

"""
mutable struct MyFibonacciSequenceModel <: AbstractSequenceModel

    n::Int64 # largest Fibonacci index; the complete mapping has n + 1 entries
    sequence::Dict{Int64, Int64} # map from each mathematical index k to F_k

    # Leave both fields undefined so the construction example can assign them explicitly.
    MyFibonacciSequenceModel() = new();
end

"""
    MyForLoopIterationModel <: AbstractIterationModel

Fieldless marker that selects the for-loop implementation.
"""
struct MyForLoopIterationModel <: AbstractIterationModel
    MyForLoopIterationModel() = new();
end

"""
    MyWhileLoopIterationModel <: AbstractIterationModel

Fieldless marker that selects the while-loop implementation.
"""
struct MyWhileLoopIterationModel <: AbstractIterationModel
    MyWhileLoopIterationModel() = new();
end

Next, define a public `fibonacci!(...)` method that validates inputs and manages the state of `MyFibonacciSequenceModel`. It delegates the calculation to internal `_fibonacci(...)` methods.

Julia does not enforce private methods. A leading underscore (`_`) marks a method as internal by convention. Separating the public interface from the calculation allows the implementation to change without changing how callers use the function.

> __Method versus function__: A __function__ is a named operation. A __method__ is one implementation of that function for a particular set of argument types. With multiple dispatch, Julia selects a method using the types of the arguments.

The next cells implement this design.

In [ ]:
# Internal implementations selected by the iteration-model argument.
function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyForLoopIterationModel)

    @info "Debug message: We are using the for loop iteration model"

    # Copy the requested largest index and build a fresh index-to-value mapping.
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();

    # Seed F_0. For n == 0, store that complete result before returning.
    sequence[0] = 0;
    if (n == 0)
        sequencemodel.sequence = sequence;
        return nothing;
    end
    sequence[1] = 1;

    # Seed F_1 and compute F_2 through F_n from their two predecessors.
    for i ∈ 2:n
        sequence[i] = sequence[i-1] + sequence[i-2]
    end

    # Store the completed mapping in the mutable model; the method returns no value.
    sequencemodel.sequence = sequence;
    return nothing;
end

function _fibonacci(sequencemodel::MyFibonacciSequenceModel, iterationmodel::MyWhileLoopIterationModel)

    @info "Debug message: We are using the while loop iteration model"

    # Copy the requested largest index and build a fresh index-to-value mapping.
    n = sequencemodel.n;
    sequence = Dict{Int64, Int64}();
    
    # Start at F_0 and stop immediately after writing F_n.
    should_loop_continue = true
    i = 0;
    while (should_loop_continue == true)
       
        # Seed F_0 and F_1 explicitly; later indices use the recurrence.
        if (i == 0)
            sequence[i] = 0; 
        elseif (i == 1)
            sequence[i] = 1;
        else
            sequence[i] = sequence[i - 1] + sequence[i - 2]
        end

        # Advance to the next mathematical Fibonacci index.
        i += 1;

        # Stop once F_n has been written; the next index exceeds the request.
        if (i>n)
            should_loop_continue = false;
        end
    end
    
    # Store the completed mapping in the mutable model; the method returns no value.
    sequencemodel.sequence = sequence;
    return nothing;
end

# Public mutating interface shared by both loop implementations.
"""
    fibonacci!(sequencemodel::MyFibonacciSequenceModel;
        iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel -> Nothing

Populate `sequencemodel.sequence` with `F_0` through `F_n`. The iteration-model
type selects the internal loop implementation by multiple dispatch.

# Arguments
- `sequencemodel::MyFibonacciSequenceModel`: Mutable state whose `n` field is
  the largest Fibonacci index to compute.
- `iterationmodel::T`: Fieldless marker selecting the loop implementation.
  The default is `MyForLoopIterationModel()`.

The function returns `nothing` after updating `sequencemodel` in place.
"""
function fibonacci!(sequencemodel::MyFibonacciSequenceModel; 
    iterationmodel::T = MyForLoopIterationModel()) where T <: AbstractIterationModel
    
    # The state model can represent only nonnegative Fibonacci indices.
    @assert sequencemodel.n >= 0 "Hmmm. Major Malfunction - the argument `n` must be a non-negative integer";

    # Dispatch on iterationmodel to select the for-loop or while-loop method.
    _fibonacci(sequencemodel, iterationmodel);
    return nothing;
end;

Let's start by building the composite types to represent the Fibonacci sequence and its properties. We'll define a `MyFibonacciSequenceModel` type to hold the Fibonacci numbers; let's save this in the `my_sequence_model::MyFibonacciSequenceModel` variable.

In [ ]:
my_sequence_model = let
    
    # Build initially empty state for the sequence F_0 through F_10.
    n = 10; # largest Fibonacci index, giving n + 1 sequence entries
    model = MyFibonacciSequenceModel();

    # The zero-argument constructor leaves both fields undefined, so assign each
    # field before any caller attempts to read it.
    model.n = n;
    model.sequence = Dict{Int64, Int64}();

    # The let block returns this initialized model to my_sequence_model.
    model
end

Hold on to something, this next part is going to get weird! The sequence field of the `MyFibonacciSequenceModel` model should be initialized to an empty dictionary of type `Dict{Int,Int}`, i.e., its length (number of things it holds) should be `0`. Is this true?

In [ ]:
# Verify the precondition that no Fibonacci values have been stored yet.
@assert length(my_sequence_model.sequence) == 0

Ok, so now let's call our public `fibonacci!(...)` method and see what happens:

In [ ]:
# Use the default for-loop marker to populate the model in place.
fibonacci!(my_sequence_model);

That seems to have worked, but how do we get the results (the public `fibonacci!(...)` method doesn't return anything)? Also, why is there a `!` at the end of the function name? 
* _Mutating methods_: In Julia, a `!` at the end of a function name indicates that the function _modifies its arguments_ in some way that will be visible after the method execution has ended. In this case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.
* _Is there something magical about the `!` character_? No, adding the `!` character to the end of a function name is _not magic_. It's just a convention to help identify functions that may change state or data outside the local scope of the function. In this particular case, the `fibonacci!` method modifies the `my_sequence_model` by populating its `sequence` field with Fibonacci numbers.

If that is true, we should see the results stored in the `my_sequence_model.sequence` variable. Let's check this by computing the length of the sequence field, and checking that it is larger than `0` (i.e., it has some values in it).

In [ ]:
# Verify the mutation populated the model's sequence mapping.
@assert length(my_sequence_model.sequence) > 0

The other interesting thing from that call was that [the `@info` message](https://docs.julialang.org/en/v1/stdlib/Logging/#Logging.@logmsg) for the for loop got called. We didn't provide an implementation type, how did that work?

> __Optional keyword arguments__: The `iterationmodel::T` optional argument defaults to an instance of `MyForLoopIterationModel`. If we wanted to use our while loop implementation, should we pass in a `MyWhileLoopIterationModel` instance?

Let's check this out by calling the `fibonacci!` method again, but this time passing in a `MyWhileLoopIterationModel` instance as the optional argument. We should see the debug message for the while loop get called instead of the one for the for loop.

In [ ]:
# Select the while-loop implementation and replace the sequence in place.
fibonacci!(my_sequence_model, iterationmodel = MyWhileLoopIterationModel());

In [ ]:
# Display the index-to-value mapping produced by the while-loop implementation.
my_sequence_model.sequence

### Arguments and multiple dispatch
The calls above reach the same public `fibonacci!(...)` method, but `_fibonacci(...)` uses either the `for` or `while` implementation. How does Julia choose between them?

> __Multiple dispatch__: Julia selects a method using the runtime types of all positional arguments. The type of `iterationmodel` therefore selects the matching `_fibonacci(...)` method.

Julia supports three common argument forms:
* __Positional arguments__ are supplied in the order defined by the function signature. The caller must provide the required number of values, and their types must match an applicable method.
* __Optional positional arguments__ occupy a fixed position but have a default value. A caller may omit the argument to use the default or provide a value in that position to override it.
* __Keyword arguments__ appear after a semicolon in the signature and are passed by name. They can have default values, and naming them at the call site makes the purpose of each value clear without relying on argument order.

Keyword arguments do __not__ participate in dispatch, so two methods cannot differ only in the type of a keyword argument. `iterationmodel` is a keyword argument to `fibonacci!(...)`, but the public method passes it as a positional argument to `_fibonacci(...)`. Dispatch occurs on that second call.

See the [Julia methods documentation](https://docs.julialang.org/en/v1/manual/methods/#Methods) for more details.
___

## Looking ahead to Lab
In Lab `L2b` we continue to explore these ideas with a function that runs, raises no errors, but still returns the wrong answer. How do we catch this type of error? 

___

## Summary
A function is not just a rule for computing a value; it is an interface, and its argument types, return type, errors, and mutation behavior are all part of what it promises.

> __Key Takeaways:__
>
> * **The signature is the contract:** Argument types, the declared return type, the errors a function raises, and whether it mutates its input are the parts a caller depends on, so they deserve as much thought as the algorithm. The exclamation point on a mutating name such as `fibonacci!` is convention rather than magic, but it announces a promise the type signature alone cannot express.
> * **Early returns keep base cases visible:** Handling the trivial inputs first and returning immediately leaves the main body free to express the general case, which is easier to read and easier to test. The starter Fibonacci function settles its two base cases before the recurrence begins, so the loop can assume that two preceding values always exist.
> * **Dispatch replaces type branching:** Julia selects a method from the runtime types of the positional arguments, so alternative implementations of one operation live in separate methods rather than in one function full of type checks. Fieldless marker types make the choice explicit at the call site, and the public method stays a stable interface while the internal implementations remain free to change.

The functions you write from here on are the units the rest of the course composes: labs test them, later weeks import them, and problem sets extend them.
___